# The three models, on one real case

One referred loan application from the sample pack, run through everything we
call on NVIDIA Build:

| Role | Model | Why |
|---|---|---|
| Retriever | `nvidia/nemotron-3-embed-1b` | Finds which parts of the case file to read |
| Assistant | `nvidia/nemotron-3.5-lightning-30b-a3b` | Writes the briefing from retrieved text |
| Judge | `nvidia/nemotron-3-ultra-550b-a55b` | Grades readability — never completeness |

Then our own check — no model involved — asks the only question a validator cares
about: **did the briefing state the facts the decision turned on?**

Nemotron 3 Super is not here. Its free endpoint is deprecated on 2 October 2026,
five days before the final.

In [1]:
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv

from evidence.adapters.nvidia_build import MODELS, chat
from evidence.adapters.rag import as_prompt, retrieve
from evidence.checks import run_checks
from evidence.contracts.item import BenchmarkItem

load_dotenv()
PACK = Path("../packs/underwriter-sample")
print(json.dumps(MODELS, indent=2))

{
  "assistant": "nvidia/nemotron-3.5-lightning-30b-a3b",
  "judge": "nvidia/nemotron-3-ultra-550b-a55b",
  "embed": "nvidia/nemotron-3-embed-1b"
}


## 1. The case

Generated, not real. A hidden repayment-capacity tier drove the numbers; the tier was
dropped before the file was written. The assistant sees three documents: the application,
the bureau report, and an extract of the lending policy. The policy is where the 40%
limit lives — the assistant is not expected to know it from nowhere.

In [2]:
items = [BenchmarkItem.model_validate_json(line) for line in (PACK / "items.jsonl").open()]
item = next(i for i in items if "APP000044" in i.item_id)
for doc in item.context:
    print(f"── {doc.renderer} ──")
    print(doc.content)

── application_form ──
# Personal Loan Application

**Reference:** APP000044
**Received:** 31 March 2026

## Applicant

| Field | Value |
|---|---|
| Title | Mr |
| Age band | 55-64 |
| Employment type | self_employed |
| Employer | Northgate Retail |
| Time in current role | 16 months |
| Postcode district | BS7 |

## Income and outgoings

| Field | Value |
|---|---|
| Gross annual income | £15,922 |
| Income verified | Yes |
| Existing monthly credit commitments | £316 |
| Dependants | 3 |

## Facility requested

| Field | Value |
|---|---|
| Amount | £9,323 |
| Term | 36 months |
| Stated purpose | debt consolidation |
| Indicative monthly instalment | £314 |

── bureau_summary ──
# Credit Bureau Summary

**Reference:** APP000044
**Bureau:** Northgate Credit Reference
**Search date:** 31 March 2026

## Score

**652** *(range 0–999)*

## File

| Field | Value |
|---|---|
| Credit file opened | December 2020 (75 months) |
| Accounts on file | 5 |
| Searches, last 6 months | 1 |

## Pa

## 2. What the marking key says — computed before any model ran

This is the sealed half of the item. The assistant never sees it.

In [3]:
g = item.grading
print("outcome        :", g.disposition)
print("drivers        :", [g.driver_labels[r] for r in g.driver_refs])
print("must surface   :", [g.omission_labels[r] for r in g.omission_refs])
print("decoys (0 wt)  :", g.decoy_refs)
print("would flip it  :", [f.model_dump() for f in g.flip_refs])

outcome        : refer
drivers        : ['debt-to-income ratio of 47% exceeds the 40% policy limit']
must surface   : ['debt-to-income ratio of 47% exceeds the 40% policy limit', 'the 40% debt-to-income policy limit']
decoys (0 wt)  : ['age_band', 'delinquencies_24m', 'delinquency_recency_months', 'dependants', 'employer_name', 'employment_stability', 'postcode_district', 'purpose', 'tenure_months', 'title']
would flip it  : [{'ref': 'gross_annual', 'direction': 'increase'}]


## 3. Retrieval — the assistant selects what to read

Each document is split on its section headings, embedded as `passage`, and the
question is embedded as `query`. Only the top four sections go to the assistant.
This is the deployment shape, and it introduces a failure mode worth recording: if
the section holding the ratio is never retrieved, the assistant cannot state it.

In [4]:
t0 = time.perf_counter()
retrieved = retrieve(item.prompt, item.context, k=4)
print(f"retrieval took {time.perf_counter()-t0:.1f}s\n")
for r in retrieved:
    print(f"{r.score:.3f}  {r.chunk_id}")
print("\n── what the assistant is handed ──")
print(as_prompt(retrieved))

retrieval took 1.3s

0.341  lending_policy#3
0.282  lending_policy#2
0.214  lending_policy#1
0.164  application_form#0

── what the assistant is handed ──
[lending_policy#3]
## Underwriter briefing standard

A briefing must state the reason for review, the applicable threshold, and what
would need to change for the application to fall within policy.

---

[lending_policy#2]
## Credit history

| Rule | Threshold |
|---|---|
| Bureau score | Below 600 requires underwriter review |
| Missed payments | Any missed payment within the last 12 months requires underwriter review |
| Credit file age | Under 24 months is considered a thin file |

---

[lending_policy#1]
## Affordability

| Rule | Threshold |
|---|---|
| Total monthly debt service, as a share of gross monthly income | Must not exceed **40%** |
| Debt service includes | Existing credit commitments plus the proposed instalment |
| Income basis | Gross annual income, verified where available |

Applications above the 40% threshold re

## 4. The assistant writes the briefing

Nemotron 3.5 Lightning, temperature 0, seed pinned. It gets the retrieved sections
only. Every field returned here is what the run manifest records.

In [5]:
a = chat("assistant", system=item.prompt, user=as_prompt(retrieved), max_tokens=700)
print(f"{a.model_id}  prompt={a.prompt_version}  {a.latency_ms/1000:.0f}s  "
      f"in={a.tokens_in} out={a.tokens_out}\n")
print(a.text.strip())

nvidia/nemotron-3.5-lightning-30b-a3b  prompt=0f7c3b98eca20e37  10s  in=334 out=347

### Case Summary for Underwriter Review

**Reason for Referral:**
This application has been referred for manual underwriter review because the automated rules could not settle the file. Specifically, the applicant’s **Total Debt Service Ratio (DSR) exceeds the policy threshold of 40%**, and the applicant has a **thin credit file** (age under 24 months).

**What the File Shows – Against the Applicant:**

*   **Affordability Breach:** The applicant’s total monthly debt service is **45.2%** of their gross monthly income. This exceeds the maximum allowable threshold of 40% outlined in the Affordability policy. The debt service calculation includes existing credit commitments plus the proposed instalment.
*   **Credit History:** The applicant’s credit file is **18 months old**, classifying it as a "thin file" under the Credit History policy. Additionally, there is **one missed payment** recorded within the 

## 5. The judge grades readability

Nemotron 3 Ultra with thinking on. A different model from the one under test, and
larger — a judge should not be worse than the thing it grades. It is asked two
things a formula cannot check: could an underwriter understand this, and can they
tell what would need to change. **It is not asked about completeness.**

In [6]:
RUBRIC = (
    "You are grading an underwriter briefing for readability only. Answer with a JSON "
    'object {"intelligible": 0-2, "actionable": 0-2, "reason": "..."}. '
    "intelligible: could an underwriter understand it without re-reading the file. "
    "actionable: can they tell what would need to change for a different outcome. "
    "Do not grade whether the briefing is complete or correct; that is checked elsewhere."
)
j = chat("judge", system=RUBRIC, user=f"BRIEFING:\n{a.text}", max_tokens=600)
print(f"{j.model_id}  prompt={j.prompt_version}  {j.latency_ms/1000:.0f}s\n")
print(j.text.strip())

nvidia/nemotron-3-ultra-550b-a55b  prompt=9b683e462193b07b  22s

{"intelligible": 2, "actionable": 2, "reason": "The briefing is clearly structured with distinct sections for referral reason, negative factors, and specific remediation steps. Language is plain, jargon-free, and quantified (e.g., 45.2% DSR, 18-month file, 40% threshold). An underwriter can grasp the issues and required changes in a single pass without re-reading."}


## 6. The check — no model involved

`material_omission` opens the marking key and looks for each required fact in the
briefing. Exact match first, then a similarity match that is flagged for audit, then
missing. It passes only at full marks — a briefing that surfaces two of three material
facts still misleads the underwriter.

In [7]:
(chk,) = run_checks(["material_omission"], output=a.text, item=item)
print(f"material_omission → {'PASS' if chk.passed else 'FAIL'}   score {chk.score:.2f}   "
      f"needs_audit={chk.needs_audit}")
print(chk.detail, "\n")
for e in chk.evidence:
    mark = "found  " if e["matched"] else "MISSING"
    via = f" via {e['method']}" if e.get("method") else ""
    print(f"  [{mark}] {g.omission_labels[e['ref']]}{via}")

material_omission → PASS   score 1.00   needs_audit=True
all 2 material fact(s) surfaced (some by similarity — needs audit) 

  [found  ] debt-to-income ratio of 47% exceeds the 40% policy limit via similarity
  [found  ] the 40% debt-to-income policy limit via exact


## 7. Was it retrieval or the model?

Three facts, one from each document. For each: is it in the file, was it retrieved,
is it in the briefing. The transcript records the retrieved chunks so this question
can always be answered after the fact.

In [8]:
retrieved_text = " ".join(r.text for r in retrieved)
file_text = " ".join(d.content for d in item.context)
probes = {"40% limit (policy)": "40%", "instalment £314 (application)": "£314",
          "bureau score 652 (bureau report)": "652"}
print(f"{'fact':36} {'in file':>8} {'retrieved':>10} {'in briefing':>12}")
for label, needle in probes.items():
    print(f"{label:36} {str(needle in file_text):>8} {str(needle in retrieved_text):>10} "
          f"{str(needle in a.text):>12}")

print("\nNumbers the briefing states that appear nowhere in the case file:")
num = r"\d+(?:,\d{3})*(?:\.\d+)?"
file_nums = set(re.findall(num, file_text))
for n in sorted(set(re.findall(num, a.text)) - file_nums, key=lambda x: float(x.replace(",", ""))):
    print(f"  {n}")

fact                                  in file  retrieved  in briefing
40% limit (policy)                       True       True         True
instalment £314 (application)            True      False        False
bureau score 652 (bureau report)         True      False        False

Numbers the briefing states that appear nowhere in the case file:
  2
  18
  45.2


## What this shows

- **Three models, one endpoint, everything pinned.** Model id, prompt hash, parameters,
  latency and tokens recorded per call — the run manifest is built from these.
- **The judge and the check answer different questions.** The judge says whether the
  briefing reads well. The check says whether it told the underwriter what mattered.
- **RAG failed, and the framework saw it.** Retrieval ranked the policy sections above the
  data — the question reads like a policy question — so the assistant was handed rules and
  no numbers. It wrote a confident briefing anyway, with a ratio and a file-age claim that
  appear nowhere in the case. The judge gave it full marks. The omission check passed,
  because the breach *was* stated. Section 7 is what catches it: the numbers it used are
  not in the file. That is `numeric_fidelity`, the next check to port.
- **Why this is the point.** A briefing can be readable, complete on the facts we asked
  for, and still fabricated. No single check is enough; the pack runs all of them, and the
  transcript records what was retrieved so the failure can be attributed.
- **Retrieval fix for Week 2:** retrieve per document rather than across all chunks, so
  every source contributes, and query with the case fields rather than the task prompt.
- **Latency** was 8–10 s per call today against 100–150 s yesterday — the free endpoint
  varies. The runner stays resumable regardless.